# Wilcoxon Signed-Rank Tests — teacher_csboost vs All Models

Parses **already-computed** Wilcoxon results from the experiment notebooks (`gmsc.ipynb`, `churn.ipynb`, `upsell.ipynb`) and assembles a **compact thesis-ready table** for `teacher_csboost` against every other model on **AEC** and **AUROC**.

**No re-running of experiment notebooks is required** — results are read directly from saved cell outputs.

In [1]:
import json
import io
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 40)

REPO = Path.cwd().resolve()

NOTEBOOK_MAP = {
    'GMSC':  REPO / 'gmsc.ipynb',
    'Churn': REPO / 'churn.ipynb',
    'Upsell': REPO / 'upsell.ipynb',
}

TARGET = 'teacher_csboost'
METRICS_TO_EXTRACT = ['AEC', 'AUROC']
ALPHA = 0.05

print(f'Repo: {REPO}')

Repo: /Users/lea/final cost sensitive 


## 1. Parse Wilcoxon tables from experiment notebooks

In [2]:
def _read_html(html):
    for flavor in ('lxml', 'bs4', 'html5lib', None):
        try:
            return pd.read_html(io.StringIO(html), flavor=flavor)
        except Exception:
            continue
    return []


def extract_wilcoxon_tables(nb_path, metrics):
    """Parse Wilcoxon HTML tables from the experiment notebook's Wilcoxon cell.
    Returns a dict {metric_name: DataFrame}."""
    with open(nb_path, encoding='utf-8') as f:
        nb = json.load(f)

    tables = {}
    for cell in nb.get('cells', []):
        src = ''.join(cell.get('source', []))
        if 'pairwise_wilcoxon' not in src.lower() and 'holm_bonferroni' not in src.lower():
            continue
        outputs = cell.get('outputs', [])
        current_metric = None
        for out in outputs:
            if out.get('output_type') == 'stream':
                text = ''.join(out.get('text', []))
                for m in metrics:
                    if f'— {m} ' in text and 'Wilcoxon' in text:
                        current_metric = m
                        break
                continue
            if current_metric and 'text/html' in (out.get('data') or {}):
                html = ''.join(out['data']['text/html'])
                dfs = _read_html(html)
                if dfs and len(dfs[0]) > 0:
                    tables[current_metric] = dfs[0]
                current_metric = None
        if tables:
            break
    return tables


all_wilcox = []
for ds_name, nb_path in NOTEBOOK_MAP.items():
    if not nb_path.exists():
        print(f'WARNING: {nb_path.name} not found, skipping {ds_name}')
        continue
    tables = extract_wilcoxon_tables(nb_path, METRICS_TO_EXTRACT)
    for metric, df in tables.items():
        for c in list(df.columns):
            if str(c).startswith('Unnamed'):
                df = df.drop(columns=[c])
        tc_rows = df[df['target'].astype(str).str.contains(TARGET) |
                      df['baseline'].astype(str).str.contains(TARGET)].copy()
        tc_rows['Dataset'] = ds_name
        tc_rows['Metric'] = metric
        all_wilcox.append(tc_rows)
    print(f'{ds_name}: extracted {len(tables)} metric tables')

wilcox_df = pd.concat(all_wilcox, ignore_index=True)
print(f'\nTotal rows: {len(wilcox_df)}')
print(f'Metrics: {wilcox_df["Metric"].unique().tolist()}')
print(f'Datasets: {wilcox_df["Dataset"].unique().tolist()}')

GMSC: extracted 2 metric tables
Churn: extracted 0 metric tables
Upsell: extracted 0 metric tables

Total rows: 8
Metrics: ['AEC', 'AUROC']
Datasets: ['GMSC']


## 2. Normalise direction

Ensure `teacher_csboost` is always the **target** (left side) of the comparison.

In [3]:
def normalise_direction(df, target=TARGET):
    """Ensure target is always in the 'target' column; flip rows where it appears as 'baseline'."""
    out = df.copy()
    mask = out['baseline'].astype(str).str.contains(target)
    if mask.any():
        flipped = out.loc[mask].copy()
        flipped['target'], flipped['baseline'] = flipped['baseline'].values, flipped['target'].values
        flipped['target_mean'], flipped['baseline_mean'] = flipped['baseline_mean'].values, flipped['target_mean'].values
        flipped['mean_diff'] = -flipped['mean_diff']
        flipped['effect_size_r'] = -flipped['effect_size_r']
        out.loc[mask] = flipped
    return out


wilcox_df = normalise_direction(wilcox_df)
wilcox_df = wilcox_df.rename(columns={'baseline': 'Opponent'})
print(f'All comparisons are now: {TARGET} (target) vs Opponent')
print(f'Opponents: {sorted(wilcox_df["Opponent"].unique())}')

All comparisons are now: teacher_csboost (target) vs Opponent
Opponents: ['boost', 'csboost', 'cslogit', 'logit']


## 3. AEC results

In [4]:
aec = wilcox_df[wilcox_df['Metric'] == 'AEC'].copy()
cols = ['Dataset', 'Opponent', 'target_mean', 'baseline_mean', 'mean_diff',
        'effect_size_r', 'p_value', 'p_adjusted', 'result']
cols = [c for c in cols if c in aec.columns]

print(f'Wilcoxon signed-rank: {TARGET} vs each model — AEC (lower = better)')
print('Negative mean_diff = teacher_csboost has lower (better) AEC')
print('Holm-Bonferroni corrected within each dataset\n')
display(aec[cols].sort_values(['Dataset', 'mean_diff']))

Wilcoxon signed-rank: teacher_csboost vs each model — AEC (lower = better)
Negative mean_diff = teacher_csboost has lower (better) AEC
Holm-Bonferroni corrected within each dataset



,Dataset,Opponent,target_mean,baseline_mean,mean_diff,effect_size_r,p_value,p_adjusted,result
3,GMSC,logit,359.137385,716.799410,-357.662025,-1.0,0.001953,0.054688,* better
0,GMSC,boost,359.137385,605.306103,-246.168718,-1.0,0.001953,0.054688,* better
2,GMSC,cslogit,359.137385,393.292908,-34.155524,-1.0,0.001953,0.054688,* better
1,GMSC,csboost,359.137385,368.434667,-9.297283,-1.0,0.001953,0.054688,* better


## 4. AUROC results

In [5]:
auroc = wilcox_df[wilcox_df['Metric'] == 'AUROC'].copy()
cols = ['Dataset', 'Opponent', 'target_mean', 'baseline_mean', 'mean_diff',
        'effect_size_r', 'p_value', 'p_adjusted', 'result']
cols = [c for c in cols if c in auroc.columns]

print(f'Wilcoxon signed-rank: {TARGET} vs each model — AUROC (higher = better)')
print('Positive mean_diff = teacher_csboost has higher (better) AUROC')
print('Holm-Bonferroni corrected within each dataset\n')
display(auroc[cols].sort_values(['Dataset', 'mean_diff'], ascending=[True, False]))

Wilcoxon signed-rank: teacher_csboost vs each model — AUROC (higher = better)
Positive mean_diff = teacher_csboost has higher (better) AUROC
Holm-Bonferroni corrected within each dataset



,Dataset,Opponent,target_mean,baseline_mean,mean_diff,effect_size_r,p_value,p_adjusted,result
7,GMSC,logit,0.841867,0.705788,0.136079,1.000000,0.001953,0.054688,* better
6,GMSC,cslogit,0.841867,0.804429,0.037437,1.000000,0.001953,0.054688,* better
5,GMSC,csboost,0.841867,0.844847,-0.002980,-0.672727,0.064453,0.386719,n.s.
4,GMSC,boost,0.841867,0.856099,-0.014232,-1.000000,0.001953,0.054688,* worse


## 5. Compact thesis table

One row per opponent, one column per dataset. Each cell shows `mean_diff (p_adj) significance`.

In [6]:
def make_thesis_table(df, metric):
    sub = df[df['Metric'] == metric].copy()
    sub['cell'] = sub.apply(
        lambda r: f"{r['mean_diff']:+.2f} (p={r['p_adjusted']:.3f})"
                  f"{' **' if r['p_adjusted'] < 0.05 else (' *' if r['p_adjusted'] < 0.10 else '')}",
        axis=1,
    )
    pivot = sub.pivot(index='Opponent', columns='Dataset', values='cell')
    for ds in ['GMSC', 'Churn', 'Upsell']:
        if ds not in pivot.columns:
            pivot[ds] = '—'
    return pivot[['GMSC', 'Churn', 'Upsell']]


print(f'Table: AEC difference ({TARGET} minus opponent)')
print('Negative = teacher_csboost better | ** p<0.05 | * p<0.10\n')
display(make_thesis_table(wilcox_df, 'AEC'))

print(f'\nTable: AUROC difference ({TARGET} minus opponent)')
print('Positive = teacher_csboost better | ** p<0.05 | * p<0.10\n')
display(make_thesis_table(wilcox_df, 'AUROC'))

Table: AEC difference (teacher_csboost minus opponent)
Negative = teacher_csboost better | ** p<0.05 | * p<0.10



Dataset,GMSC,Churn,Upsell
Opponent,,,
boost,-246.17 (p=0.055) *,—,—
csboost,-9.30 (p=0.055) *,—,—
cslogit,-34.16 (p=0.055) *,—,—
logit,-357.66 (p=0.055) *,—,—



Table: AUROC difference (teacher_csboost minus opponent)
Positive = teacher_csboost better | ** p<0.05 | * p<0.10



Dataset,GMSC,Churn,Upsell
Opponent,,,
boost,-0.01 (p=0.055) *,—,—
csboost,-0.00 (p=0.387),—,—
cslogit,+0.04 (p=0.055) *,—,—
logit,+0.14 (p=0.055) *,—,—


## 6. Summary

In [7]:
for metric in METRICS_TO_EXTRACT:
    sub = wilcox_df[wilcox_df['Metric'] == metric]
    n_total = len(sub)
    n_sig = (sub['p_adjusted'] < ALPHA).sum()
    lower_better = metric in ('AEC', 'Cost', 'Brier')
    if lower_better:
        n_better = (sub['mean_diff'] < 0).sum()
    else:
        n_better = (sub['mean_diff'] > 0).sum()
    print(f'{metric}: {TARGET} directionally better in {n_better}/{n_total} comparisons, '
          f'{n_sig}/{n_total} statistically significant at alpha={ALPHA}')

AEC: teacher_csboost directionally better in 4/4 comparisons, 0/4 statistically significant at alpha=0.05
AUROC: teacher_csboost directionally better in 2/4 comparisons, 0/4 statistically significant at alpha=0.05
